# CXR-Recall — Arm 1 (no-retrieval baseline) on Kaggle

Self-contained notebook version of the local `cxr_recall` repo, adapted to run on Kaggle against the **Indiana University Chest X-ray Collection** dataset (attach it via the **+ Add Data** panel — search "Indiana University chest x-ray" or "Chest X-rays Indiana University").

Also turn on **Settings → Accelerator → GPU** before running.

This mirrors `src/data/label_schema.py`, `src/data/splits.py`, `src/data/iu_xray.py`, `src/models/*`, and `train.py` from the local repo, inlined into one notebook. It builds a coarse MeSH/Problems-tag label vocabulary for a pipeline smoke test — not the thesis's final labeled result (that needs MIMIC-CXR/CheXpert once credentialed).

**Run the first few cells first** — they print what's actually in the attached dataset, since Kaggle mirrors of this collection vary slightly in column naming. If the auto-detected column names below don't match what gets printed, edit the `UID_COL` / `FINDINGS_COL` / etc. constants in the manifest-building cell — everything downstream is unaffected.

In [ ]:
import torch
print('torch:', torch.__version__, ' cuda available:', torch.cuda.is_available())

In [ ]:
import random
import numpy as np

# Reproducibility: without this, model init (classification heads, evidence
# encoder layers) and DataLoader shuffle order are unseeded, so identical code
# produces genuinely different results between runs -- confirmed directly when
# Arm 1 alone scored 0.7335 in one run and 0.7192 in another. set_seed() is
# called again right before each training loop below, so each arm (Arm 1,
# Arm 3c, Arm 3c-frozen) is independently reproducible on its own -- useful
# later for multi-seed significance testing (see RESULTS.md "Next steps").
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print('seed set to', SEED)

## 1. Locate the attached dataset

In [ ]:
import os

INPUT_DIR = '/kaggle/input'
candidates = os.listdir(INPUT_DIR)
assert candidates, "No dataset attached — use '+ Add Data' and search for the Indiana University chest x-ray collection first."
print('attached datasets:', candidates)

DATASET_DIR = os.path.join(INPUT_DIR, candidates[0])
print('using:', DATASET_DIR)
for root, dirs, files in os.walk(DATASET_DIR):
    print(root, '->', dirs, files[:5])

In [ ]:
import glob

csv_files = glob.glob(os.path.join(DATASET_DIR, '**', '*.csv'), recursive=True)
png_dirs = sorted(set(os.path.dirname(p) for p in glob.glob(os.path.join(DATASET_DIR, '**', '*.png'), recursive=True)))
print('csv files:', csv_files)
print('folders containing pngs:', png_dirs)

In [ ]:
import pandas as pd

for f in csv_files:
    df = pd.read_csv(f)
    print(f, df.shape)
    print(df.columns.tolist())
    print(df.head(2))
    print('---')

## 2. Build the manifest
Adjust the `*_COL` constants below if the printed column names above differ from these guesses (the common `raddar/chest-xrays-indiana-university` mirror uses `uid`, `findings`, `impression`, `Problems` in the reports CSV, and `uid`, `filename` in the projections CSV).

In [ ]:
reports_csv = [f for f in csv_files if 'report' in os.path.basename(f).lower()]
projections_csv = [f for f in csv_files if 'projection' in os.path.basename(f).lower()]
assert reports_csv and projections_csv, 'Could not auto-find reports/projections CSVs -- inspect csv_files above and set paths manually.'

reports_df = pd.read_csv(reports_csv[0])
proj_df = pd.read_csv(projections_csv[0])

UID_COL, FINDINGS_COL, IMPRESSION_COL, TAGS_COL = 'uid', 'findings', 'impression', 'Problems'
PROJ_UID_COL, FILENAME_COL = 'uid', 'filename'

merged = proj_df.merge(reports_df, left_on=PROJ_UID_COL, right_on=UID_COL, suffixes=('', '_report'))

def build_report_text(row):
    parts = [str(row.get(FINDINGS_COL, '')), str(row.get(IMPRESSION_COL, ''))]
    return ' '.join(p for p in parts if p and p.lower() != 'nan').strip()

def build_tags(row):
    raw = row.get(TAGS_COL, '')
    if pd.isna(raw):
        return ''
    sep = ';' if ';' in str(raw) else ','
    return ';'.join(t.strip() for t in str(raw).split(sep) if t.strip())

manifest = pd.DataFrame({
    'image_path': merged[FILENAME_COL],
    'patient_id': merged[UID_COL].astype(str),
    'report_text': merged.apply(build_report_text, axis=1),
    'mesh_tags': merged.apply(build_tags, axis=1),
})

IMAGE_ROOT = png_dirs[0]
print('manifest rows:', len(manifest), ' image root:', IMAGE_ROOT)
manifest.head()

## 3. Label vocabulary (from `mesh_tags` / `Problems`)

In [ ]:
from collections import Counter
import numpy as np

def build_mesh_vocab(tag_lists, top_k=14):
    counts = Counter(tag for tags in tag_lists for tag in tags)
    return [tag for tag, _ in counts.most_common(top_k)]

def mesh_tags_to_vector(tags, vocab):
    tag_set = set(tags)
    return np.array([1.0 if v in tag_set else 0.0 for v in vocab], dtype=np.float32)

## 4. Patient-level split (no leakage across train/val/test)

In [ ]:
def patient_level_split(df, patient_id_col, ratios=(0.8, 0.1, 0.1), seed=42):
    patients = sorted(df[patient_id_col].unique())
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    n_train, n_val = int(n * ratios[0]), int(n * ratios[1])
    train_p = set(patients[:n_train])
    val_p = set(patients[n_train:n_train + n_val])
    test_p = set(patients[n_train + n_val:])
    return (df[df[patient_id_col].isin(train_p)],
            df[df[patient_id_col].isin(val_p)],
            df[df[patient_id_col].isin(test_p)])

train_df, val_df, test_df = patient_level_split(manifest, 'patient_id')
print('split sizes:', len(train_df), len(val_df), len(test_df))

train_tags = [t.split(';') if t else [] for t in train_df['mesh_tags']]
label_vocab = build_mesh_vocab(train_tags, top_k=14)
print('label vocab:', label_vocab)

## 5. Dataset + model (Arm 1: no-retrieval baseline)

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class IUXrayDataset(Dataset):
    def __init__(self, df, image_root, transform, label_vocab):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform
        self.label_vocab = label_vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.image_root, row['image_path'])).convert('RGB')
        image = self.transform(image)
        tags = row['mesh_tags'].split(';') if row['mesh_tags'] else []
        label = mesh_tags_to_vector(tags, self.label_vocab)
        return image, label

In [ ]:
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights

class NoRetrievalClassifier(nn.Module):
    def __init__(self, num_labels, pretrained=True):
        super().__init__()
        weights = ViT_B_16_Weights.DEFAULT if pretrained else None
        self.backbone = vit_b_16(weights=weights)
        self.backbone.heads = nn.Identity()
        self.head = nn.Linear(768, num_labels)

    def forward(self, images):
        return self.head(self.backbone(images))

transform = ViT_B_16_Weights.DEFAULT.transforms()
train_ds = IUXrayDataset(train_df, IMAGE_ROOT, transform, label_vocab)
val_ds = IUXrayDataset(val_df, IMAGE_ROOT, transform, label_vocab)
test_ds = IUXrayDataset(test_df, IMAGE_ROOT, transform, label_vocab)  # held out: never used for early stopping or model selection
print('train/val/test sizes:', len(train_ds), len(val_ds), len(test_ds))

## 6. Setup: data loaders, loss, training helper

No training happens in this cell — `NoRetrievalClassifier` and `IUXrayDataset` were already defined in Section 5. This just builds `train_loader`/`val_loader`, the loss, and the `macro_auroc` metric that every later training loop reuses. (A smoke test used to run here to validate the pipeline before spending real compute; that's already been proven out in prior sessions — see `RESULTS.md` — so this notebook goes straight to real training in Section 8.)

In [ ]:
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE, LR = 16, 1e-4

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

criterion = nn.BCEWithLogitsLoss()

def macro_auroc(y_true, y_score):
    scores = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        scores.append(roc_auc_score(y_true[:, i], y_score[:, i]))
    return float(np.mean(scores)) if scores else float('nan')

def held_out_auroc(forward_fn, loader):
    """One pass over a held-out loader with the model already in eval() mode.
    forward_fn(batch_tuple) -> logits. Every loader in this notebook yields
    labels at position 1. Used for the TEST numbers: val is what every arm's
    early stopping selects on, so val AUROC is an optimistically-selected
    number; test is evaluated exactly once per arm, on the reloaded best
    checkpoint, and never influences any training or selection decision."""
    all_labels, all_scores = [], []
    with torch.no_grad():
        for batch in loader:
            all_scores.append(torch.sigmoid(forward_fn(batch)).cpu().numpy())
            all_labels.append(batch[1].numpy())
    return macro_auroc(np.concatenate(all_labels), np.concatenate(all_scores))

print('setup complete -- device:', device, ' batch size:', BATCH_SIZE, ' lr:', LR)

## 7. Retrieval utilities (function definitions only)

Defines `extract_embeddings`, `build_index`, `search`, `label_agreement_at_k`, and `mean_reciprocal_rank` — the FAISS-based retrieval machinery Arm 2 and Arm 3c both need. Nothing runs yet; these are just function definitions, exercised for real starting in Section 8. (A smoke-test retrieval check — including a random-baseline comparison and a qualitative spot-check — used to run here too; already validated in prior sessions, see `RESULTS.md`.)

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss

@torch.no_grad()
def extract_embeddings(backbone, loader, device):
    backbone.eval()
    all_emb, all_labels = [], []
    for images, labels in loader:
        emb = backbone(images.to(device)).cpu().numpy()
        all_emb.append(emb)
        all_labels.append(labels.numpy())
    return np.concatenate(all_emb), np.concatenate(all_labels)

In [ ]:
def build_index(embeddings):
    embeddings = embeddings.astype(np.float32).copy()
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index

def search(index, queries, k):
    queries = queries.astype(np.float32).copy()
    faiss.normalize_L2(queries)
    return index.search(queries, k)

def label_agreement_at_k(query_labels, retrieved_labels):
    if retrieved_labels.shape[0] == 0:
        return float('nan')
    shares = (retrieved_labels * query_labels).sum(axis=1) > 0
    return float(shares.mean())

def mean_reciprocal_rank(query_labels, retrieved_labels):
    shares = (retrieved_labels * query_labels).sum(axis=1) > 0
    hits = np.nonzero(shares)[0]
    return 0.0 if len(hits) == 0 else 1.0 / (hits[0] + 1)

K = 5
print('retrieval utilities defined -- k =', K)

## 8. Real training run (early stopping)

This is the first cell that actually trains anything in this notebook — Sections 1–7 were data setup and function/loader definitions only. It trains Arm 1 for real (up to 20 epochs, stopping once val AUROC hasn't improved for 5 epochs), then rebuilds Arm 2's retrieval index on that trained encoder, for use by Arm 3c below.

The pipeline-validation step that used to run before this (a cheap 3-epoch smoke test + retrieval sanity check) has been removed — it was already proven out across multiple prior sessions, documented in `RESULTS.md`, and re-running it every time just to re-confirm known-working code wasn't worth the time. One consequence worth knowing: the cells just above (Sections 6–7) were recently restructured to drop their own training/execution and keep only the definitions Section 8 needs — that restructuring itself hasn't been executed end-to-end yet, so if something errors out immediately below, that's the first place to look.

In [ ]:
set_seed(SEED)  # re-seed so this arm's own init/shuffling is independently reproducible

import copy

EPOCHS_REAL = 20
PATIENCE = 5

model = NoRetrievalClassifier(num_labels=len(label_vocab)).to(device)  # fresh weights, not continuing the smoke-test run
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_auroc, best_state, epochs_without_improvement = -1.0, None, 0

for epoch in range(1, EPOCHS_REAL + 1):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    total_loss /= max(len(train_ds), 1)

    model.eval()
    all_labels, all_scores = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            probs = torch.sigmoid(model(images.to(device))).cpu().numpy()
            all_labels.append(labels.numpy())
            all_scores.append(probs)
    val_auroc = macro_auroc(np.concatenate(all_labels), np.concatenate(all_scores))
    print(f'[Arm 1 real] epoch {epoch}/{EPOCHS_REAL}  train_loss={total_loss:.4f}  val_macro_auroc={val_auroc:.4f}')

    if val_auroc > best_auroc:
        best_auroc, best_state, epochs_without_improvement = val_auroc, copy.deepcopy(model.state_dict()), 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'no val improvement for {PATIENCE} epochs -- stopping early at epoch {epoch}')
            break

model.load_state_dict(best_state)
print(f'best Arm 1 val_macro_auroc = {best_auroc:.4f}  (best checkpoint reloaded into `model`)')
model.eval()
test_auroc = held_out_auroc(lambda b: model(b[0].to(device)), test_loader)
print(f'Arm 1 TEST macro_auroc = {test_auroc:.4f}  (held out, evaluated once on the reloaded best checkpoint)')
torch.save({'model_state_dict': model.state_dict(), 'label_vocab': label_vocab}, '/kaggle/working/arm1_real_best.pt')

In [ ]:
train_loader_seq = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print('re-extracting train embeddings from the real-trained encoder...')
train_embeddings, train_labels = extract_embeddings(model.backbone, train_loader_seq, device)
print('re-extracting val embeddings...')
val_embeddings, val_labels = extract_embeddings(model.backbone, val_loader, device)

index = build_index(train_embeddings)
val_scores, neighbor_idx = search(index, val_embeddings, K)  # val_scores: (n_val, K) cosine similarities -- kept for Section 12's confidence gating, not just neighbor_idx

agreements = [label_agreement_at_k(val_labels[i], train_labels[neighbor_idx[i]]) for i in range(len(val_embeddings))]
mrrs = [mean_reciprocal_rank(val_labels[i], train_labels[neighbor_idx[i]]) for i in range(len(val_embeddings))]
print(f'[Arm 2 real] label_agreement@{K}={np.mean(agreements):.4f}  MRR={np.mean(mrrs):.4f}')

rng = np.random.default_rng(42)
random_agreements = [label_agreement_at_k(val_labels[i], train_labels[rng.choice(len(train_labels), size=K, replace=False)])
                      for i in range(len(val_embeddings))]
print(f'[Arm 2 real] random baseline label_agreement@{K}={np.mean(random_agreements):.4f}')

print('extracting test embeddings...')
test_embeddings, test_labels = extract_embeddings(model.backbone, test_loader, device)
test_scores, test_neighbor_idx = search(index, test_embeddings, K)  # test queries retrieve from the TRAIN index only
test_agreements = [label_agreement_at_k(test_labels[i], train_labels[test_neighbor_idx[i]]) for i in range(len(test_embeddings))]
test_mrrs = [mean_reciprocal_rank(test_labels[i], train_labels[test_neighbor_idx[i]]) for i in range(len(test_embeddings))]
print(f'[Arm 2 TEST] label_agreement@{K}={np.mean(test_agreements):.4f}  MRR={np.mean(test_mrrs):.4f}')

faiss.write_index(index, '/kaggle/working/arm2_retrieval_real.index')
np.save('/kaggle/working/arm2_train_embeddings_real.npy', train_embeddings)
np.save('/kaggle/working/arm2_train_labels_real.npy', train_labels)

In [ ]:
# Train-vs-train retrieval for evidence-augmented training below.
#
# Self-exclusion is by patient_id (study), NOT just by row index. 85% of IU X-ray
# studies have 2+ views (frontal + lateral) that share one report and one label
# vector, so excluding only the query's own row left its study-mate in the pool
# -- a neighbor whose label token IS the query's label. Val/test queries can never
# retrieve a study-mate (the split is patient-level), so index-only exclusion
# trained the evidence encoder under a regime it never sees at eval time. The
# LEAKAGE CHECK line reports how often that actually fired.
train_pids = train_df['patient_id'].to_numpy()  # same row order as train_ds / train_embeddings (IUXrayDataset resets index in order)
MAX_VIEWS_PER_STUDY = int(train_df.groupby('patient_id').size().max())
train_scores_raw, train_neighbor_idx_raw = search(index, train_embeddings, K + MAX_VIEWS_PER_STUDY)

train_neighbor_idx, train_top1_sim, same_study_hits = [], [], 0
for i, (row, row_scores) in enumerate(zip(train_neighbor_idx_raw, train_scores_raw)):
    index_only = [j for j in row if j != i][:K]  # what the old exclusion would have returned
    same_study_hits += int(any(train_pids[j] == train_pids[i] for j in index_only))
    keep = [(j, s) for j, s in zip(row, row_scores) if j != i and train_pids[j] != train_pids[i]][:K]
    assert len(keep) == K, f'query {i}: only {len(keep)} non-self, non-same-study neighbors in the top {K + MAX_VIEWS_PER_STUDY}'
    train_neighbor_idx.append([j for j, _ in keep])
    train_top1_sim.append(keep[0][1])
train_neighbor_idx = np.array(train_neighbor_idx)
train_top1_sim = np.array(train_top1_sim, dtype=np.float32)  # top-1 similarity per train query -- Section 12 calibrates its threshold on this (train side), not on val

print('train_neighbor_idx shape:', train_neighbor_idx.shape)
print(f'LEAKAGE CHECK: {same_study_hits}/{len(train_pids)} ({same_study_hits / len(train_pids):.1%}) train queries had a '
      f'same-study view in their top-{K} under index-only exclusion -- now excluded.')


In [ ]:
class EvidenceAugmentedDatasetV2(Dataset):
    """Wraps IUXrayDataset, additionally returning each item's k retrieved
    neighbors' label vectors AND image embeddings -- feeds Arm 3c below.
    (The 'V2' name is kept only because Section 11's cells already reference
    it; Arm 3b, which originally introduced it, is no longer built here.)"""
    def __init__(self, base_df, image_root, transform, label_vocab, evidence_pool_labels, evidence_pool_embeddings, neighbor_idx):
        self.base = IUXrayDataset(base_df, image_root, transform, label_vocab)
        self.evidence_pool_labels = evidence_pool_labels
        self.evidence_pool_embeddings = evidence_pool_embeddings
        self.neighbor_idx = neighbor_idx

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        image, label = self.base[idx]
        idxs = self.neighbor_idx[idx]
        evidence_labels = self.evidence_pool_labels[idxs]
        evidence_images = self.evidence_pool_embeddings[idxs]
        return image, label, torch.tensor(evidence_labels, dtype=torch.float32), torch.tensor(evidence_images, dtype=torch.float32)

In [ ]:
train_ds_arm3b = EvidenceAugmentedDatasetV2(train_df, IMAGE_ROOT, transform, label_vocab, train_labels, train_embeddings, train_neighbor_idx)
val_ds_arm3b = EvidenceAugmentedDatasetV2(val_df, IMAGE_ROOT, transform, label_vocab, train_labels, train_embeddings, neighbor_idx)
train_loader_arm3b = DataLoader(train_ds_arm3b, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader_arm3b = DataLoader(val_ds_arm3b, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_ds_arm3b = EvidenceAugmentedDatasetV2(test_df, IMAGE_ROOT, transform, label_vocab, train_labels, train_embeddings, test_neighbor_idx)
test_loader_arm3b = DataLoader(test_ds_arm3b, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print('evidence-augmented dataset sizes:', len(train_ds_arm3b), len(val_ds_arm3b), len(test_ds_arm3b))

## 9. Arm 3c — joint attention (image and label as separate tokens)

Skips straight past Arm 3 (label-only) and Arm 3b (early fusion) — both documented in `RESULTS.md`, not rebuilt here. Arm 3b's design fused each neighbor's image embedding and label vector into one blended vector *before* attention ran — the attention mechanism only ever saw k pre-blended tokens, never the two modalities as independently weightable signals. This section keeps them as **2k separately attendable tokens**, tagged with a learned modality embedding (image vs. label) and a learned neighbor-position embedding (so the model still knows which image/label pair belongs to the same neighbor). The query can now learn to trust a specific neighbor's image while discounting that same neighbor's label, or vice versa — something early fusion structurally couldn't do.

Worth watching for: 2k attendable items is more capacity than a k-token design, on the same small dataset — exactly the kind of change that could reproduce Arm 3's late-epoch overfitting collapse if it's too much capacity for the data available.

Reuses `train_loader_arm3b` / `val_loader_arm3b` — built two cells above (the name is historical, kept only because it's what these cells reference; no Arm 3b model is trained in this notebook).

In [ ]:
class EvidenceEncoderV3(nn.Module):
    """Joint attention: image and label tokens stay separate (2k total),
    tagged with a modality embedding and a neighbor-position embedding."""
    def __init__(self, num_labels, embed_dim, num_heads=4, image_embed_dim=None, max_k=16):
        super().__init__()
        image_embed_dim = image_embed_dim or embed_dim
        self.label_proj = nn.Linear(num_labels, embed_dim)
        self.image_proj = nn.Linear(image_embed_dim, embed_dim)
        self.modality_embed = nn.Embedding(2, embed_dim)      # 0 = image, 1 = label
        self.neighbor_pos_embed = nn.Embedding(max_k, embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads=num_heads, batch_first=True)

    def forward(self, query_embedding, evidence_labels, evidence_image_embeddings):
        k = evidence_labels.shape[1]
        device = evidence_labels.device

        label_embed = self.label_proj(evidence_labels)            # (batch, k, embed_dim)
        image_embed = self.image_proj(evidence_image_embeddings)  # (batch, k, embed_dim)

        pos = self.neighbor_pos_embed(torch.arange(k, device=device)).unsqueeze(0)  # (1, k, embed_dim)
        image_type = self.modality_embed.weight[0]
        label_type = self.modality_embed.weight[1]

        image_tok = image_embed + pos + image_type
        label_tok = label_embed + pos + label_type
        evidence_tokens = torch.cat([image_tok, label_tok], dim=1)  # (batch, 2k, embed_dim) -- separately attendable

        query = query_embedding.unsqueeze(1)
        context, _weights = self.attention(query, evidence_tokens, evidence_tokens)
        return context.squeeze(1)


class RetrievalFusionClassifierV3(nn.Module):
    def __init__(self, num_labels, pretrained=True, image_embed_dim=768, max_k=16):
        super().__init__()
        weights = ViT_B_16_Weights.DEFAULT if pretrained else None
        self.backbone = vit_b_16(weights=weights)
        self.backbone.heads = nn.Identity()
        self.evidence_encoder = EvidenceEncoderV3(num_labels, 768, image_embed_dim=image_embed_dim, max_k=max_k)
        self.head = nn.Linear(768, num_labels)

    def forward(self, images, evidence_labels, evidence_image_embeddings, evidence_confidence=None):
        # evidence_confidence: optional (batch,) tensor, hard 0.0/1.0 from a
        # similarity-threshold cutoff (Section 12) -- 0.0 zeroes this sample's
        # evidence contribution entirely, falling back to a pure vision-only
        # prediction for it; omitted (default), behavior is unchanged from
        # every earlier Arm 3c/3c-frozen cell above.
        query_embedding = self.backbone(images)
        evidence_context = self.evidence_encoder(query_embedding, evidence_labels, evidence_image_embeddings)
        if evidence_confidence is not None:
            fused = query_embedding + evidence_context * evidence_confidence.unsqueeze(-1)
        else:
            fused = query_embedding + evidence_context
        return self.head(fused)

In [ ]:
set_seed(SEED)  # re-seed so this arm's own init/shuffling is independently reproducible

model_arm3c = RetrievalFusionClassifierV3(num_labels=len(label_vocab), image_embed_dim=train_embeddings.shape[1], max_k=K).to(device)
optimizer_arm3c = torch.optim.Adam(model_arm3c.parameters(), lr=LR)

best_auroc3c, best_state3c, epochs_without_improvement3c = -1.0, None, 0

for epoch in range(1, EPOCHS_REAL + 1):
    model_arm3c.train()
    total_loss = 0.0
    for images, labels, ev_labels, ev_images in train_loader_arm3b:
        images, labels = images.to(device), labels.to(device)
        ev_labels, ev_images = ev_labels.to(device), ev_images.to(device)
        optimizer_arm3c.zero_grad()
        loss = criterion(model_arm3c(images, ev_labels, ev_images), labels)
        loss.backward()
        optimizer_arm3c.step()
        total_loss += loss.item() * images.size(0)
    total_loss /= max(len(train_ds_arm3b), 1)

    model_arm3c.eval()
    all_labels3c, all_scores3c = [], []
    with torch.no_grad():
        for images, labels, ev_labels, ev_images in val_loader_arm3b:
            probs = torch.sigmoid(model_arm3c(images.to(device), ev_labels.to(device), ev_images.to(device))).cpu().numpy()
            all_labels3c.append(labels.numpy())
            all_scores3c.append(probs)
    val_auroc3c = macro_auroc(np.concatenate(all_labels3c), np.concatenate(all_scores3c))
    print(f'[Arm 3c real] epoch {epoch}/{EPOCHS_REAL}  train_loss={total_loss:.4f}  val_macro_auroc={val_auroc3c:.4f}')

    if val_auroc3c > best_auroc3c:
        best_auroc3c, best_state3c, epochs_without_improvement3c = val_auroc3c, copy.deepcopy(model_arm3c.state_dict()), 0
    else:
        epochs_without_improvement3c += 1
        if epochs_without_improvement3c >= PATIENCE:
            print(f'no val improvement for {PATIENCE} epochs -- stopping early at epoch {epoch}')
            break

model_arm3c.load_state_dict(best_state3c)
print(f'best Arm 3c val_macro_auroc = {best_auroc3c:.4f}')
model_arm3c.eval()
test_auroc3c = held_out_auroc(lambda b: model_arm3c(b[0].to(device), b[2].to(device), b[3].to(device)), test_loader_arm3b)
print(f'Arm 3c TEST macro_auroc = {test_auroc3c:.4f}')
torch.save({'model_state_dict': model_arm3c.state_dict(), 'label_vocab': label_vocab}, '/kaggle/working/arm3c_real_best.pt')

In [ ]:
model_arm3c.eval()
real_scores_c, zeroed_all_scores_c, zeroed_images_only_scores_c = [], [], []
with torch.no_grad():
    for images, labels, ev_labels, ev_images in val_loader_arm3b:
        images_d = images.to(device)
        ev_labels_d = ev_labels.to(device)
        ev_images_d = ev_images.to(device)
        real_scores_c.append(torch.sigmoid(model_arm3c(images_d, ev_labels_d, ev_images_d)).cpu().numpy())
        zeroed_all_scores_c.append(torch.sigmoid(model_arm3c(images_d, torch.zeros_like(ev_labels_d), torch.zeros_like(ev_images_d))).cpu().numpy())
        zeroed_images_only_scores_c.append(torch.sigmoid(model_arm3c(images_d, ev_labels_d, torch.zeros_like(ev_images_d))).cpu().numpy())

real_scores_c = np.concatenate(real_scores_c)
zeroed_all_scores_c = np.concatenate(zeroed_all_scores_c)
zeroed_images_only_scores_c = np.concatenate(zeroed_images_only_scores_c)

shift_all_c = np.mean(np.abs(real_scores_c - zeroed_all_scores_c))
shift_images_only_c = np.mean(np.abs(real_scores_c - zeroed_images_only_scores_c))
print(f'[Arm 3c real] full evidence-ablation mean |shift| = {shift_all_c:.4f}')
print(f'[Arm 3c real] image-specific ablation (labels kept, images zeroed) mean |shift| = {shift_images_only_c:.4f}')

In [ ]:
print('=== Arm 1 (no retrieval) vs Arm 3c (joint attention) ===')
print(f'Arm 1  (no retrieval)    val_macro_auroc = {best_auroc:.4f}   TEST = {test_auroc:.4f}')
print(f'Arm 3c (joint attention) val_macro_auroc = {best_auroc3c:.4f}   TEST = {test_auroc3c:.4f}   full shift = {shift_all_c:.4f}   image-only shift = {shift_images_only_c:.4f}')
print(f'  test-set margin (Arm 3c - Arm 1) = {test_auroc3c - test_auroc:+.4f}   (val margin = {best_auroc3c - best_auroc:+.4f}; val is selected-on, test is not)')
print()
if best_auroc3c > best_auroc:
    print(f'Arm 3c beats Arm 1 by {best_auroc3c - best_auroc:.4f} AUROC -- H1 supported with joint attention.')
else:
    print(f'Arm 3c still trails Arm 1 by {best_auroc - best_auroc3c:.4f}.')
print('(Arm 3 label-only and Arm 3b early-fusion results are documented in RESULTS.md;')
print('this notebook no longer reproduces them, to keep reruns fast.)')

## 10. Arm 3c-frozen — the encoder-sharing fix

Arm 3c (and Arm 3/3b before it, documented in `RESULTS.md`) each trained their own encoder end-to-end, which confounds the AUROC comparison against Arm 1: some of Arm 3c's gap-closing could reflect its encoder adapting differently under fusion-mediated gradients, not only the fusion mechanism itself. This section removes that confound directly: `model_arm3c_frozen`'s encoder is loaded from **Arm 1's exact trained weights** (`model.backbone`, the best checkpoint from Section 8) and frozen — no gradients reach it at all. Only the evidence encoder and classification head train.

With the encoder now byte-identical to Arm 1's, **any remaining AUROC gap between Arm 1 and Arm 3c-frozen is attributable purely to the fusion mechanism** — this is the clean, isolated test the proposal's original design called for.

One expectation to set before running: this could plausibly score *lower* than the unfrozen Arm 3c (0.7318), and that would not mean "fusion matters less than we thought" — it would mean the frozen features are less flexible, since Arm 1's encoder was never trained with fusion in mind. The comparison that actually matters here is **Arm 1 vs. Arm 3c-frozen**, not Arm 3c vs. Arm 3c-frozen.

In [ ]:
set_seed(SEED)  # re-seed so this arm's own init/shuffling is independently reproducible

model_arm3c_frozen = RetrievalFusionClassifierV3(num_labels=len(label_vocab), image_embed_dim=train_embeddings.shape[1], max_k=K).to(device)
model_arm3c_frozen.backbone.load_state_dict(model.backbone.state_dict())  # Arm 1's exact trained weights
for param in model_arm3c_frozen.backbone.parameters():
    param.requires_grad = False
model_arm3c_frozen.backbone.eval()

trainable_params = [p for p in model_arm3c_frozen.parameters() if p.requires_grad]
optimizer_arm3c_frozen = torch.optim.Adam(trainable_params, lr=LR)

best_auroc3cf, best_state3cf, epochs_without_improvement3cf = -1.0, None, 0

for epoch in range(1, EPOCHS_REAL + 1):
    model_arm3c_frozen.train()
    model_arm3c_frozen.backbone.eval()  # keep the frozen encoder out of train mode even though the rest of the model is training
    total_loss = 0.0
    for images, labels, ev_labels, ev_images in train_loader_arm3b:
        images, labels = images.to(device), labels.to(device)
        ev_labels, ev_images = ev_labels.to(device), ev_images.to(device)
        optimizer_arm3c_frozen.zero_grad()
        loss = criterion(model_arm3c_frozen(images, ev_labels, ev_images), labels)
        loss.backward()
        optimizer_arm3c_frozen.step()
        total_loss += loss.item() * images.size(0)
    total_loss /= max(len(train_ds_arm3b), 1)

    model_arm3c_frozen.eval()
    all_labels3cf, all_scores3cf = [], []
    with torch.no_grad():
        for images, labels, ev_labels, ev_images in val_loader_arm3b:
            probs = torch.sigmoid(model_arm3c_frozen(images.to(device), ev_labels.to(device), ev_images.to(device))).cpu().numpy()
            all_labels3cf.append(labels.numpy())
            all_scores3cf.append(probs)
    val_auroc3cf = macro_auroc(np.concatenate(all_labels3cf), np.concatenate(all_scores3cf))
    print(f'[Arm 3c-frozen real] epoch {epoch}/{EPOCHS_REAL}  train_loss={total_loss:.4f}  val_macro_auroc={val_auroc3cf:.4f}')

    if val_auroc3cf > best_auroc3cf:
        best_auroc3cf, best_state3cf, epochs_without_improvement3cf = val_auroc3cf, copy.deepcopy(model_arm3c_frozen.state_dict()), 0
    else:
        epochs_without_improvement3cf += 1
        if epochs_without_improvement3cf >= PATIENCE:
            print(f'no val improvement for {PATIENCE} epochs -- stopping early at epoch {epoch}')
            break

model_arm3c_frozen.load_state_dict(best_state3cf)
print(f'best Arm 3c-frozen val_macro_auroc = {best_auroc3cf:.4f}')
model_arm3c_frozen.eval()
test_auroc3cf = held_out_auroc(lambda b: model_arm3c_frozen(b[0].to(device), b[2].to(device), b[3].to(device)), test_loader_arm3b)
print(f'Arm 3c-frozen TEST macro_auroc = {test_auroc3cf:.4f}')
torch.save({'model_state_dict': model_arm3c_frozen.state_dict(), 'label_vocab': label_vocab}, '/kaggle/working/arm3c_frozen_real_best.pt')

In [ ]:
model_arm3c_frozen.eval()
real_scores_f, zeroed_all_scores_f, zeroed_images_only_scores_f = [], [], []
with torch.no_grad():
    for images, labels, ev_labels, ev_images in val_loader_arm3b:
        images_d = images.to(device)
        ev_labels_d = ev_labels.to(device)
        ev_images_d = ev_images.to(device)
        real_scores_f.append(torch.sigmoid(model_arm3c_frozen(images_d, ev_labels_d, ev_images_d)).cpu().numpy())
        zeroed_all_scores_f.append(torch.sigmoid(model_arm3c_frozen(images_d, torch.zeros_like(ev_labels_d), torch.zeros_like(ev_images_d))).cpu().numpy())
        zeroed_images_only_scores_f.append(torch.sigmoid(model_arm3c_frozen(images_d, ev_labels_d, torch.zeros_like(ev_images_d))).cpu().numpy())

real_scores_f = np.concatenate(real_scores_f)
zeroed_all_scores_f = np.concatenate(zeroed_all_scores_f)
zeroed_images_only_scores_f = np.concatenate(zeroed_images_only_scores_f)

shift_all_f = np.mean(np.abs(real_scores_f - zeroed_all_scores_f))
shift_images_only_f = np.mean(np.abs(real_scores_f - zeroed_images_only_scores_f))
print(f'[Arm 3c-frozen real] full evidence-ablation mean |shift| = {shift_all_f:.4f}')
print(f'[Arm 3c-frozen real] image-specific ablation (labels kept, images zeroed) mean |shift| = {shift_images_only_f:.4f}')

In [ ]:
print("=== Encoder-sharing fix: Arm 1 vs Arm 3c (own encoder) vs Arm 3c-frozen (Arm 1's exact encoder) ===")
print(f'Arm 1               (no retrieval)               val_macro_auroc = {best_auroc:.4f}   TEST = {test_auroc:.4f}')
print(f'Arm 3c              (own encoder, joint attn)     val_macro_auroc = {best_auroc3c:.4f}   TEST = {test_auroc3c:.4f}   val gap to Arm 1 = {best_auroc - best_auroc3c:.4f}')
print(f'Arm 3c-frozen       (Arm 1 encoder, joint attn)   val_macro_auroc = {best_auroc3cf:.4f}   TEST = {test_auroc3cf:.4f}   val gap to Arm 1 = {best_auroc - best_auroc3cf:.4f}')
print(f'TEST margins (held out, not selected on): Arm 3c - Arm 1 = {test_auroc3c - test_auroc:+.4f}   Arm 3c-frozen - Arm 1 = {test_auroc3cf - test_auroc:+.4f}')
print(f'                    full shift = {shift_all_f:.4f}   image-only shift = {shift_images_only_f:.4f}')
print()
if best_auroc3cf >= best_auroc:
    print(f"Arm 3c-frozen matches or beats Arm 1 ({best_auroc3cf:.4f} vs {best_auroc:.4f}) with a byte-identical, frozen encoder -- the fusion mechanism alone accounts for this, with the encoder-adaptation confound fully removed.")
else:
    gap = best_auroc - best_auroc3cf
    print(f"Arm 3c-frozen trails Arm 1 by {gap:.4f} with the encoder confound fully removed -- this is the cleanest isolated estimate yet of what the fusion mechanism alone contributes (or costs).")
    if gap < (best_auroc - best_auroc3c):
        print("This gap is SMALLER than unfrozen Arm 3c's gap -- the frozen encoder actually did better than Arm 3c's own independently-trained one for this task.")
    else:
        print("This gap is LARGER than unfrozen Arm 3c's gap -- some of Arm 3c's earlier improvement may have come from its encoder adapting during training, not the fusion mechanism alone.")

## 11. Phase 2 — text modality (vision-only vs. text-only vs. fusion, decomposed ablation)

Adds the report text (`report_text`, already in the manifest since Section 2 but unused by every arm so far) as a second modality alongside the image. Fusion is a residual sum — `image_embedding + text_embedding`, same style as Arm 3's `query_embedding + evidence_context` — followed by the same linear head shape as every prior arm.

**Deliberately one trained model, not three.** Rather than training vision-only / text-only / fusion as three separate models, this trains a single fusion model once and ablates each modality at eval time (zero it out, measure the prediction shift) — exactly the decomposed-ablation pattern already validated for Arm 3's retrieved evidence. Two reasons: it reuses ablation machinery this project already trusts, and it sidesteps a real circularity risk a separately-trained text-only model would walk straight into.

**The circularity risk:** IU X-ray's `mesh_tags` labels are NLM's own MeSH indexing *of the report text itself*. A text-only model scoring well is expected almost by construction — it would partly be reading its own label source, not demonstrating that report text carries independent clinical signal. The lexical-overlap check below puts a number on how serious this is for this dataset before any model is trained, so the eventual ablation results can be read against a concrete baseline rather than an abstract worry. Vision-only doesn't need retraining — it's just Arm 1 (Section 8's `best_auroc`).

**One infra note:** `Bio_ClinicalBERT` downloads from the HuggingFace Hub on first use — make sure this notebook's Internet toggle is on (Settings panel, right sidebar) before running the cells below.

In [ ]:
# Phase 2 attempt 1 (Sections 11a/11b) is superseded by the contrastive redesign in
# cxr_recall_phase2_kaggle.ipynb and already has real numbers (RESULTS.md). Its training
# cost ~1.2 h of the 3.8 h audit rerun, so it is OFF by default; flip to True only to
# reproduce it. Section 12 does not depend on it.
RUN_PHASE2_ATTEMPT1 = False

# How much of any text-modality signal would be circular? For each (report,
# label) pair restricted to the label vocab, check whether the tag's own text
# literally appears in that report. MeSH normalizes surface forms (e.g. plurals,
# synonyms), so this naive substring match undercounts true overlap -- treat it
# as a floor on circularity, not a ceiling.

def label_lexical_overlap(df, vocab):
    total, present = 0, 0
    for _, row in df.iterrows():
        text = str(row['report_text']).lower()
        tags = row['mesh_tags'].split(';') if row['mesh_tags'] else []
        for tag in tags:
            if tag in vocab:
                total += 1
                if tag.lower() in text:
                    present += 1
    return present / total if total else float('nan')

overlap = label_lexical_overlap(train_df, label_vocab)
print(f'label-vocab tags whose own text appears literally in their report: {overlap:.1%} (naive substring floor)')

### 11a. Multimodal dataset + model

In [ ]:
if RUN_PHASE2_ATTEMPT1:  # superseded Phase 2 attempt 1 -- skipped by default on reruns, see cell above
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoModel, AutoTokenizer

    class IUXrayMultimodalDataset(Dataset):
        def __init__(self, df, image_root, transform, label_vocab):
            self.df = df.reset_index(drop=True)
            self.image_root = image_root
            self.transform = transform
            self.label_vocab = label_vocab

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            image = Image.open(os.path.join(self.image_root, row['image_path'])).convert('RGB')
            image = self.transform(image)
            text = '' if pd.isna(row['report_text']) else str(row['report_text'])
            tags = row['mesh_tags'].split(';') if row['mesh_tags'] else []
            label = mesh_tags_to_vector(tags, self.label_vocab)
            return image, text, label


    class TextEncoder(nn.Module):
        def __init__(self, model_name='emilyalsentzer/Bio_ClinicalBERT'):
            super().__init__()
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModel.from_pretrained(model_name)
            self.embed_dim = self.model.config.hidden_size

        def forward(self, texts, device, max_length=256):
            tokens = self.tokenizer(texts, padding=True, truncation=True,
                                     max_length=max_length, return_tensors='pt').to(device)
            output = self.model(**tokens)
            return output.last_hidden_state[:, 0, :]  # [CLS] embedding


    class MultimodalFusionClassifier(nn.Module):
        """use_image=False / use_text=False zero that modality's contribution
        before the residual sum -- and skip calling its encoder entirely, so
        ablating one modality over a full val pass doesn't pay for the other
        encoder's forward pass."""
        def __init__(self, num_labels, pretrained=True, text_model_name='emilyalsentzer/Bio_ClinicalBERT'):
            super().__init__()
            weights = ViT_B_16_Weights.DEFAULT if pretrained else None
            self.image_backbone = vit_b_16(weights=weights)
            self.image_backbone.heads = nn.Identity()
            self.image_embed_dim = 768
            self.text_encoder = TextEncoder(text_model_name)
            self.text_proj = (nn.Identity() if self.text_encoder.embed_dim == self.image_embed_dim
                               else nn.Linear(self.text_encoder.embed_dim, self.image_embed_dim))
            self.head = nn.Linear(self.image_embed_dim, num_labels)

        def forward(self, images=None, texts=None, device=None, use_image=True, use_text=True):
            if not use_image and not use_text:
                raise ValueError('at least one modality must be used')
            if use_image:
                image_embedding = self.image_backbone(images)
                batch, device = image_embedding.size(0), image_embedding.device
            if use_text:
                text_embedding = self.text_proj(self.text_encoder(texts, device))
                batch, device = text_embedding.size(0), text_embedding.device
            if not use_image:
                image_embedding = torch.zeros(batch, self.image_embed_dim, device=device)
            if not use_text:
                text_embedding = torch.zeros(batch, self.image_embed_dim, device=device)
            return self.head(image_embedding + text_embedding)

    train_ds_mm = IUXrayMultimodalDataset(train_df, IMAGE_ROOT, transform, label_vocab)
    val_ds_mm = IUXrayMultimodalDataset(val_df, IMAGE_ROOT, transform, label_vocab)

    # Smaller batch than Arm 1's 16 -- ViT + ClinicalBERT together in one forward
    # pass roughly doubles activation memory; drop this further if you hit OOM.
    MM_BATCH_SIZE = 8
    train_loader_mm = DataLoader(train_ds_mm, batch_size=MM_BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader_mm = DataLoader(val_ds_mm, batch_size=MM_BATCH_SIZE, shuffle=False, num_workers=2)
    print('multimodal train/val sizes:', len(train_ds_mm), len(val_ds_mm))


### 11b. Train the fusion model, then decomposed ablation

In [ ]:
if RUN_PHASE2_ATTEMPT1:  # superseded Phase 2 attempt 1 -- skipped by default on reruns, see cell above
    set_seed(SEED)  # re-seed so this arm's own init/shuffling is independently reproducible

    model_mm = MultimodalFusionClassifier(num_labels=len(label_vocab)).to(device)

    # ClinicalBERT gets a lower LR than the rest (image backbone + text_proj + head,
    # which keep Arm 1's LR) -- 1e-4 is tuned for ViT fine-tuning, and is aggressive
    # for BERT-family fine-tuning, which typically wants ~1e-5-5e-5.
    LR_TEXT = 2e-5
    text_params = list(model_mm.text_encoder.parameters())
    other_params = [p for p in model_mm.parameters() if not any(p is tp for tp in text_params)]
    optimizer_mm = torch.optim.Adam([
        {'params': other_params, 'lr': LR},
        {'params': text_params, 'lr': LR_TEXT},
    ])

    best_auroc_mm, best_state_mm, epochs_without_improvement_mm = -1.0, None, 0

    for epoch in range(1, EPOCHS_REAL + 1):
        model_mm.train()
        total_loss = 0.0
        for images, texts, labels in train_loader_mm:
            images, labels = images.to(device), labels.to(device)
            optimizer_mm.zero_grad()
            loss = criterion(model_mm(images, texts, device), labels)
            loss.backward()
            optimizer_mm.step()
            total_loss += loss.item() * images.size(0)
        total_loss /= max(len(train_ds_mm), 1)

        model_mm.eval()
        all_labels_mm, all_scores_mm = [], []
        with torch.no_grad():
            for images, texts, labels in val_loader_mm:
                probs = torch.sigmoid(model_mm(images.to(device), texts, device)).cpu().numpy()
                all_labels_mm.append(labels.numpy())
                all_scores_mm.append(probs)
        val_auroc_mm = macro_auroc(np.concatenate(all_labels_mm), np.concatenate(all_scores_mm))
        print(f'[Phase 2 fusion] epoch {epoch}/{EPOCHS_REAL}  train_loss={total_loss:.4f}  val_macro_auroc={val_auroc_mm:.4f}')

        if val_auroc_mm > best_auroc_mm:
            best_auroc_mm, best_state_mm, epochs_without_improvement_mm = val_auroc_mm, copy.deepcopy(model_mm.state_dict()), 0
        else:
            epochs_without_improvement_mm += 1
            if epochs_without_improvement_mm >= PATIENCE:
                print(f'no val improvement for {PATIENCE} epochs -- stopping early at epoch {epoch}')
                break

    model_mm.load_state_dict(best_state_mm)
    print(f'best Phase 2 fusion val_macro_auroc = {best_auroc_mm:.4f}')
    torch.save({'model_state_dict': model_mm.state_dict(), 'label_vocab': label_vocab}, '/kaggle/working/phase2_fusion_best.pt')


In [ ]:
if RUN_PHASE2_ATTEMPT1:  # superseded Phase 2 attempt 1 -- skipped by default on reruns, see cell above
    model_mm.eval()
    full_scores, image_only_scores, text_only_scores, all_labels_mm2 = [], [], [], []
    with torch.no_grad():
        for images, texts, labels in val_loader_mm:
            images_d = images.to(device)
            full_scores.append(torch.sigmoid(model_mm(images_d, texts, device, use_image=True, use_text=True)).cpu().numpy())
            image_only_scores.append(torch.sigmoid(model_mm(images_d, texts, device, use_image=True, use_text=False)).cpu().numpy())
            text_only_scores.append(torch.sigmoid(model_mm(images_d, texts, device, use_image=False, use_text=True)).cpu().numpy())
            all_labels_mm2.append(labels.numpy())

    y_true_mm = np.concatenate(all_labels_mm2)
    full_scores = np.concatenate(full_scores)
    image_only_scores = np.concatenate(image_only_scores)
    text_only_scores = np.concatenate(text_only_scores)

    auroc_full = macro_auroc(y_true_mm, full_scores)
    auroc_image_only = macro_auroc(y_true_mm, image_only_scores)  # fusion model, text zeroed
    auroc_text_only = macro_auroc(y_true_mm, text_only_scores)    # fusion model, image zeroed

    shift_text_ablated = np.mean(np.abs(full_scores - image_only_scores))  # how much zeroing text moves predictions
    shift_image_ablated = np.mean(np.abs(full_scores - text_only_scores))  # how much zeroing image moves predictions

    print('=== Phase 2: image vs. text vs. fusion (single trained model, decomposed ablation) ===')
    print(f'Arm 1 (vision-only, Section 8, separately trained) val_macro_auroc = {best_auroc:.4f}')
    print(f'Fusion, full (image+text)                          val_macro_auroc = {auroc_full:.4f}')
    print(f'Fusion, image-only  (text zeroed)                  val_macro_auroc = {auroc_image_only:.4f}   shift from full = {shift_text_ablated:.4f}')
    print(f'Fusion, text-only   (image zeroed)                 val_macro_auroc = {auroc_text_only:.4f}   shift from full = {shift_image_ablated:.4f}')
    print(f'label-vocab lexical overlap (circularity floor): {overlap:.1%}')
    print()
    if auroc_text_only >= auroc_image_only:
        print('Text-only ablation outperforms image-only ablation within the fusion model -- read this alongside the lexical-overlap number above before concluding text is more clinically informative than images, given the MeSH-from-report-text circularity risk.')
    else:
        print('Image-only ablation outperforms text-only ablation within the fusion model -- despite MeSH labels being derived from report text, the image still carries more of the signal the model relies on.')


## 12. Retrieval-confidence gating (similarity-threshold fallback)

Nothing so far has checked *how similar* a retrieved neighbor actually is before trusting it as evidence — `label_agreement@{K}` above is only marginally better than the random baseline, meaning most retrieved neighbors don't even share a label with the query, yet Arm 3c-frozen fuses whatever k neighbors FAISS returns regardless of match quality. This section adds a per-query similarity threshold: when a query's best (top-1) retrieved match is below the threshold, the prediction falls back to **vision-only** for that query (still a real prediction, not an abstention) and gets flagged as "no similar cases retrieved — recommend human review."

**Applied at eval time on the already-trained `model_arm3c_frozen` checkpoint (Section 10) — no retraining.** This mirrors exactly how every evidence-ablation test above already works (zero the evidence contribution post-hoc on a trained checkpoint), just driven by a real per-query similarity signal instead of researcher-imposed 100% zeroing. Train-time gating is a legitimate follow-up (Phase 1g candidate) but reopens the same encoder-adaptation confound `load_and_freeze_encoder()` exists to eliminate, so it's out of scope here.

**The threshold is calibrated on the *train* side, then applied to val/test** — two complementary methods below: a quintile-bucket table sanity-checked against a random-retrieval baseline, and a precise ROC/Youden operating point. Both are fit on `train_top1_sim` / `train_neighbor_idx` (train-vs-train retrieval, self- and same-study-excluded, from Section 8). An earlier draft fit Youden's J on `val_labels` and then measured gated AUROC on that same val set — the threshold would have been tuned on its own evaluation data. Val's bucket table is still printed, as an out-of-sample check of the calibrated cutoff, and the final gated numbers are reported on both val and the held-out test set.


In [ ]:
from sklearn.metrics import roc_curve

def label_agreement_by_similarity_bucket(top1_similarity, query_labels, retrieved_labels, num_buckets=5):
    # equal-*frequency* buckets (quantiles via sorted order), not equal-width
    # similarity ranges -- cosine similarity on a fine-tuned embedding space
    # tends to cluster, so equal-width bins would leave some buckets nearly empty
    order = np.argsort(top1_similarity)
    buckets = np.array_split(order, num_buckets)
    results = []
    for bucket_indices in buckets:
        agreements = [label_agreement_at_k(query_labels[i], retrieved_labels[i]) for i in bucket_indices]
        sims = top1_similarity[bucket_indices]
        results.append({
            'similarity_range': (float(sims.min()), float(sims.max())),
            'mean_top1_sim': float(sims.mean()),
            'mean_label_agreement': float(np.nanmean(agreements)),
            'count': len(bucket_indices),
        })
    return results

def confidence_mask(top1_similarity, threshold):
    return (top1_similarity >= threshold).astype(np.float32)

def flag_low_confidence(top1_similarity, threshold):
    below = top1_similarity < threshold
    return {
        'flagged_indices': np.nonzero(below)[0],
        'num_flagged': int(below.sum()),
        'num_total': int(len(top1_similarity)),
        'fraction_flagged': float(below.mean()) if len(top1_similarity) else float('nan'),
    }

# --- CALIBRATION: train side only (train_top1_sim / train_neighbor_idx from Section 8,
# self- and same-study-excluded). Val/test labels are never touched here. ---
train_retrieved_labels = train_labels[train_neighbor_idx]  # (n_train, K, num_labels)
rng_cal = np.random.default_rng(42)
train_random_agreement = np.mean([label_agreement_at_k(train_labels[i], train_labels[rng_cal.choice(len(train_labels), size=K, replace=False)])
                                  for i in range(len(train_labels))])

print('--- quintile-bucket calibration on TRAIN (low similarity -> high similarity) ---')
for b in label_agreement_by_similarity_bucket(train_top1_sim, train_labels, train_retrieved_labels, num_buckets=5):
    print(f"sim {b['similarity_range'][0]:.3f}-{b['similarity_range'][1]:.3f}  "
          f"mean_top1_sim={b['mean_top1_sim']:.3f}  label_agreement={b['mean_label_agreement']:.4f}  "
          f"n={b['count']}  (train random baseline = {train_random_agreement:.4f})")

# ROC/Youden on TRAIN: is the single nearest neighbor itself label-relevant?
train_top1_is_relevant = np.array([label_agreement_at_k(train_labels[i], train_retrieved_labels[i][:1])
                                    for i in range(len(train_labels))])
fpr, tpr, roc_thresholds = roc_curve(train_top1_is_relevant, train_top1_sim)
youden_idx = np.argmax(tpr - fpr)
THRESHOLD = float(roc_thresholds[youden_idx])
print(f'\ncalibrated threshold (Youden\'s J on TRAIN top-1-neighbor relevance) = {THRESHOLD:.4f}')

# --- OUT-OF-SAMPLE CHECK on val: does the train-calibrated cutoff separate relevant from
# irrelevant retrieval on data it was not fit on? (A measurement, not a fitting step.) ---
top1_sim = val_scores[:, 0]
retrieved_labels_per_query = train_labels[neighbor_idx]  # (n_val, K, num_labels)
print('\n--- quintile-bucket table on VAL (out-of-sample, threshold NOT fit here) ---')
for b in label_agreement_by_similarity_bucket(top1_sim, val_labels, retrieved_labels_per_query, num_buckets=5):
    print(f"sim {b['similarity_range'][0]:.3f}-{b['similarity_range'][1]:.3f}  "
          f"mean_top1_sim={b['mean_top1_sim']:.3f}  label_agreement={b['mean_label_agreement']:.4f}  "
          f"n={b['count']}  (val random baseline = {np.mean(random_agreements):.4f})")
print('(if this disagrees a lot with where the bucket table crosses toward the random baseline, that mismatch is itself worth noting -- see RESULTS.md.)')

In [ ]:
def gated_eval(loader, conf):
    """conf: (n,) 0/1 mask aligned to the loader's shuffle=False iteration order."""
    model_arm3c_frozen.eval()
    scores, offset = [], 0
    with torch.no_grad():
        for images, labels, ev_labels, ev_images in loader:
            batch_conf = torch.tensor(conf[offset:offset + images.size(0)], dtype=torch.float32).to(device)
            scores.append(torch.sigmoid(model_arm3c_frozen(images.to(device), ev_labels.to(device), ev_images.to(device),
                                                            evidence_confidence=batch_conf)).cpu().numpy())
            offset += images.size(0)
    return np.concatenate(scores)

# val
conf = confidence_mask(top1_sim, THRESHOLD)
review = flag_low_confidence(top1_sim, THRESHOLD)
print(f"VAL:  {review['num_flagged']}/{review['num_total']} ({review['fraction_flagged']:.1%}) queries flagged: "
      f"no similar cases retrieved (top-1 similarity < {THRESHOLD:.4f}) -- used image-only prediction, recommend human review")
gated_scores = gated_eval(val_loader_arm3b, conf)
gated_auroc = macro_auroc(val_labels, gated_scores)
print(f'gated Arm 3c-frozen val_macro_auroc = {gated_auroc:.4f}')

# test (held out; threshold was calibrated on train, so nothing here was fit on test)
test_top1_sim = test_scores[:, 0]
test_conf = confidence_mask(test_top1_sim, THRESHOLD)
test_review = flag_low_confidence(test_top1_sim, THRESHOLD)
print(f"TEST: {test_review['num_flagged']}/{test_review['num_total']} ({test_review['fraction_flagged']:.1%}) queries flagged")
gated_test_auroc = macro_auroc(test_labels, gated_eval(test_loader_arm3b, test_conf))
print(f'gated Arm 3c-frozen TEST macro_auroc = {gated_test_auroc:.4f}')


In [ ]:
# Hybrid fallback -- the operationally honest meaning of "fall back to vision-only".
#
# The gated numbers above zero the evidence context INSIDE model_arm3c_frozen for flagged
# queries. That head was trained only ever seeing query + evidence_context, so a zero
# context is an out-of-distribution input for it -- the same failure Phase 2's gated
# fusion showed when handed a zero vector it never trained on. It is why gated AUROC in
# the audit rerun (0.6797 test) fell BELOW Arm 1 itself (0.7051): flagged queries were not
# getting a vision-only prediction, they were getting a corrupted Arm 3 prediction.
# Here, flagged queries get Arm 1's OWN prediction (a model trained without evidence),
# unflagged queries keep Arm 3c-frozen's ungated prediction. No retraining.
def arm1_scores(loader):
    model.eval()
    out = []
    with torch.no_grad():
        for batch in loader:
            out.append(torch.sigmoid(model(batch[0].to(device))).cpu().numpy())
    return np.concatenate(out)

ungated_val_scores = gated_eval(val_loader_arm3b, np.ones_like(conf))
hybrid_val_scores = np.where(conf[:, None] > 0, ungated_val_scores, arm1_scores(val_loader_arm3b))
hybrid_auroc = macro_auroc(val_labels, hybrid_val_scores)

ungated_test_scores = gated_eval(test_loader_arm3b, np.ones_like(test_conf))
hybrid_test_scores = np.where(test_conf[:, None] > 0, ungated_test_scores, arm1_scores(test_loader_arm3b))
hybrid_test_auroc = macro_auroc(test_labels, hybrid_test_scores)
print(f'hybrid (Arm 3c-frozen for confident queries, Arm 1 for flagged)  val_macro_auroc = {hybrid_auroc:.4f}   TEST = {hybrid_test_auroc:.4f}')


In [ ]:
print('=== Retrieval-confidence gating: Arm 1 vs Arm 3c-frozen (ungated) vs Arm 3c-frozen (gated) ===')
print(f'Arm 1          (no retrieval)                     val_macro_auroc = {best_auroc:.4f}   TEST = {test_auroc:.4f}')
print(f'Arm 3c-frozen  (ungated)                          val_macro_auroc = {best_auroc3cf:.4f}   TEST = {test_auroc3cf:.4f}')
print(f'Arm 3c-frozen  (gated, threshold={THRESHOLD:.4f})           val_macro_auroc = {gated_auroc:.4f}   TEST = {gated_test_auroc:.4f}')
print(f'Hybrid         (Arm 3c-frozen if confident, else Arm 1)  val_macro_auroc = {hybrid_auroc:.4f}   TEST = {hybrid_test_auroc:.4f}')
print(f"  val:  {review['num_flagged']}/{review['num_total']} ({review['fraction_flagged']:.1%}) queries fell below threshold -> vision-only fallback, flagged for human review")
print(f"  test: {test_review['num_flagged']}/{test_review['num_total']} ({test_review['fraction_flagged']:.1%}) queries flagged")
print(f'  (threshold calibrated on TRAIN retrieval only -- val and test are both out-of-sample for it)')
print()
# Verdict on TEST (held out): val is what the underlying checkpoint was selected on.
if gated_test_auroc > test_auroc3cf:
    print(f'Gating improves on ungated Arm 3c-frozen by {gated_test_auroc - test_auroc3cf:.4f} TEST AUROC -- refusing to trust weak retrieval matches helps.')
elif gated_test_auroc < test_auroc3cf:
    print(f'Gating trails ungated Arm 3c-frozen by {test_auroc3cf - gated_test_auroc:.4f} TEST AUROC -- the frozen model may already be robust to weak evidence on its own (see its evidence-ablation shift above).')
else:
    print('Gating makes no measurable TEST AUROC difference vs. ungated Arm 3c-frozen.')
print(f'(val direction for reference: gated {gated_auroc:.4f} vs ungated {best_auroc3cf:.4f})')
if hybrid_test_auroc > test_auroc3cf:
    print(f'Hybrid fallback beats ungated Arm 3c-frozen on TEST ({hybrid_test_auroc:.4f} vs {test_auroc3cf:.4f}) -- routing weak-retrieval queries to Arm 1 helps.')
else:
    print(f'Hybrid fallback does not beat ungated Arm 3c-frozen on TEST ({hybrid_test_auroc:.4f} vs {test_auroc3cf:.4f}) -- '
          'Arm 3c-frozen handles weak evidence at least as well as Arm 1 does without it; gating at this threshold buys nothing.')
print("(This is an eval-time-only comparison on the same trained checkpoint -- no retraining. "
      "A train-time-gated variant is a candidate follow-up, not built here; see RESULTS.md.)")